# Lab 2: Target Filtering - Dynamic ROI và Point-in-Polygon

Chào mừng bạn đến với Lab 2. Trong bài này, chúng ta sẽ xây dựng cơ chế lọc mục tiêu (target filtering) để xác định vật cản nào thực sự nguy hiểm.

---

## Tổng quan

Sau khi đã có vị trí và khoảng cách của các vật thể, hệ thống vẫn chưa thể đưa ra quyết định điều khiển ngay lập tức. Lý do là không phải mọi vật thể đều nằm trên quỹ đạo di chuyển của xe.

Ví dụ:
- Xe ở lane bên cạnh → không cần tránh
- Vật nằm ngoài hướng rẽ → không nguy hiểm

Vì vậy, chúng ta cần một **Region of Interest (ROI)** để xác định vùng nguy hiểm thực sự.

---

## Ý tưởng chính

Lab này gồm 2 phần:

### 1. Dynamic ROI
- ROI không cố định
- Thay đổi theo góc lái (heading angle)

### 2. Point-in-Polygon (PIP)
- Kiểm tra vật có nằm trong ROI không
- Sử dụng thuật toán Ray Casting

---

## Mục tiêu học tập

- Xây dựng ROI động theo góc lái
- Hiểu cách polygon thay đổi hình dạng
- Implement thuật toán Point-in-Polygon
- Lọc được các object nguy hiểm

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Dynamic ROI

## Lý thuyết

ROI được biểu diễn dưới dạng polygon (đa giác).

- Khi xe đi thẳng → ROI đối xứng
- Khi xe rẽ → ROI bị "bẻ cong"

Ta sẽ mô phỏng điều này bằng cách dịch chuyển đỉnh trên của polygon theo góc lái.

---

In [ ]:
def create_dynamic_roi(width, height, heading_deg):
    """
    width, height: kích thước BEV
    heading_deg: góc lái (độ)
    """

    # Chuẩn hóa heading (-1 → 1)
    norm = heading_deg / 30.0  # giả sử max góc = 30 độ

    # TODO: tính offset ngang
    offset = [...]

    # Các điểm đáy (fixed)
    bottom_left = (width * 0.3, height)
    bottom_right = (width * 0.7, height)

    # TODO: dịch chuyển top points theo offset
    top_left = [...]
    top_right = [...]

    roi = np.array([
        bottom_left,
        bottom_right,
        top_right,
        top_left
    ], dtype=np.float32)

    return roi

## Visualize ROI

In [ ]:
def plot_roi(roi, width, height):
    plt.figure()
    
    xs = list(roi[:,0]) + [roi[0,0]]
    ys = list(roi[:,1]) + [roi[0,1]]
    
    plt.plot(xs, ys, marker='o')
    plt.xlim(0, width)
    plt.ylim(height, 0)
    plt.title("Dynamic ROI")
    plt.show()


roi = create_dynamic_roi(400, 600, heading_deg=15)
plot_roi(roi, 400, 600)

# 2. Point-in-Polygon (Ray Casting)

## Lý thuyết

Để kiểm tra một điểm có nằm trong polygon hay không:

- Vẽ một tia ngang từ điểm đó
- Đếm số lần giao với cạnh polygon

Nếu số lần giao là **lẻ** → điểm nằm trong  
Nếu **chẵn** → điểm nằm ngoài  

---

In [ ]:
def point_in_polygon(x, y, polygon):
    n = len(polygon)
    inside = False

    px, py = x, y

    for i in range(n):
        x1, y1 = polygon[i]
        x2, y2 = polygon[(i + 1) % n]

        # TODO: kiểm tra tia cắt cạnh
        if ((y1 > py) != (y2 > py)):
            
            # TODO: tính giao điểm
            x_intersect = [...]
            
            # TODO: flip trạng thái
            if px < x_intersect:
                inside = [...]

    return inside

# 3. Áp dụng cho object detection

Ta sẽ:
- Lấy bottom-center của bounding box
- Kiểm tra xem điểm này có nằm trong ROI không

In [ ]:
def get_bottom_center(box):
    x1, y1, x2, y2 = box

    # TODO
    cx = [...]
    cy = [...]

    return cx, cy

In [ ]:
# Giả lập bounding boxes
boxes = [
    (150, 200, 200, 300),
    (50, 100, 100, 200),
    (250, 150, 300, 280)
]

roi = create_dynamic_roi(400, 600, heading_deg=10)

print("Objects inside ROI:")

for box in boxes:
    cx, cy = get_bottom_center(box)
    
    inside = point_in_polygon(cx, cy, roi)
    
    print(f"Box {box} -> inside: {inside}")